In [ ]:
import os
import xarray as xr
import geopandas as gpd
import numpy as np
from shapely.geometry import box, mapping
from shapely.ops import unary_union
import f# Import the gc module for garbage collection
import warnings

# Define the path to the folder containing Shapefiles
shapefile_folder = r"C:\Users\Majid\Desktop\Extreme Projections\DATAsets\muf534\ecoregions"
nc_file_path = r"C:\Users\Majid\Desktop\Extreme Projections\DATAsets\muf534\3-Sliced_ NorthAmerica\2-ACCESS-ESM1-5\combined_nex_gddp_cmip6_data_1_ACCESS_CM2_historical_r1i1p1f1_tas_NorthAmerica.nc"

# Define the path for saving output files
output_base_folder = r"C:\Users\Majid\Desktop\Extreme Projections\DATAsets\muf534\4-EcoClip\2-ACCESS-ESM1-5"

# Load the NetCDF file and set the geographic coordinate reference system
ds = xr.open_dataset(nc_file_path)
ds = ds.rio.write_crs("EPSG:4326")
print(ds)

# Define the variable to process
data_variable = 'tas'

# Calculate the grid resolution
resolution = 0.25  # Grid resolution in degrees
half_res = resolution / 2  # Half of the resolution

# Create square polygons around the center coordinates of grid cells
lon, lat = np.meshgrid(ds.lon.values, ds.lat.values)
polygons = []

for i in range(len(ds.lat)):
    for j in range(len(ds.lon)):
        # Center coordinates of the grid cell
        center_lon = lon[i, j]
        center_lat = lat[i, j]
        
        # Corners of the square around the center cell
        min_lon = center_lon - half_res
        max_lon = center_lon + half_res
        min_lat = center_lat - half_res
        max_lat = center_lat + half_res
        
        # Create a square polygon
        polygon = box(min_lon, min_lat, max_lon, max_lat)
        polygons.append(polygon)

# Create a GeoDataFrame for the grid polygons
grid_gdf = gpd.GeoDataFrame({'geometry': polygons}, crs="EPSG:4326")

# Process each Shapefile in the folder
for shapefile in os.listdir(shapefile_folder):
    if shapefile.endswith('.shp'):
        shapefile_path = os.path.join(shapefile_folder, shapefile)
        
        # Load the Shapefile
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs != "EPSG:4326":
            gdf = gdf.to_crs("EPSG:4326")
        
        # Create a mask to identify grid cells that should be retained
        mask = np.zeros(len(grid_gdf), dtype=bool)

        for i, cell in grid_gdf.iterrows():
            cell_polygon = cell.geometry
            intersection_area = 0

            for poly in gdf.geometry:
                intersection = cell_polygon.intersection(poly)
                intersection_area += intersection.area

            if intersection_area / cell_polygon.area > 0.5:
                mask[i] = True

        # Filter the grid polygons based on the mask
        filtered_polygons = grid_gdf[mask]

        if not filtered_polygons.empty:
            # Convert the filtered polygons to a single shapely geometry object
            filtered_geometry = unary_union(filtered_polygons.geometry)

            # Clip the dataset based on the filtered polygons
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                clipped = ds.rio.clip([mapping(filtered_geometry)], gdf.crs, drop=True, invert=False)

            # Extract the base name of the NetCDF file without extension
            nc_file_name = os.path.basename(nc_file_path)
            base_name = os.path.splitext(nc_file_name)[0]

            # Parse the NetCDF file name to create nested directories
            parts = base_name.split('_')

            # Ensure there are enough parts in the file name
            if len(parts) >= 10:
                # Use the eighth and tenth parts of the file name
                folder1 = parts[8]  # Eighth part of the file name (0-based index)
                folder2 = parts[10]  # Tenth part of the file name (0-based index)

                # Create the output folder path
                output_folder = os.path.join(output_base_folder, folder1, folder2)
                os.makedirs(output_folder, exist_ok=True)

                # Get the eco-region code from the Shapefile name
                eco_region_code = os.path.splitext(shapefile)[0]

                # Create the new output file name with 'ecoclipped' prefix and eco-region code
                output_file_name = f"ecoclipped_{base_name}_{eco_region_code}.nc"
                output_file_path = os.path.join(output_folder, output_file_name)
                
                # Save the clipped data to a new NetCDF file
                clipped[data_variable].to_netcdf(output_file_path)

                print(f"Processed and saved: {output_file_path}")

            # Release memory resources
            del gdf, filtered_polygons, filtered_geometry, clipped
            gc.collect()  # Garbage collection
